In [1]:
import pandas as pd
import numpy as np
import webbrowser
import os
from geopy.geocoders import ArcGIS
import time
import folium
import osmnx as ox
import networkx as nx
import pyproj
from shapely.geometry import Point

In [2]:
def get_coordinates(df):
    result_data=[]
    geolocator = ArcGIS(timeout=20)
    for _, row in df.iterrows():
        address= f"{row['Address']}, {row['City']}, {row['State']}, {row['Pin Code']}, India"
        try:
            location = geolocator.geocode(address)

            if location:
                result_data.append([
                    row['Store Name'],
                    row['Address'],
                    row['Pin Code'],
                    location.latitude,
                    location.longitude
                ])
            else:
                result_data.append([
                    row['Store Name'],
                    row['Address'],
                    row['Pin Code'],
                    None,
                    None
                ])

            # Nominatim rate limit
            time.sleep(1)

        except Exception as e:
            result_data.append([
                row['Store Name'],
                row['Address'],
                row['Pin Code'],
                None,
                None
            ])
            print(f"Error for address: {address}")
            print(e)

    return pd.DataFrame(
        result_data,
        columns=["Store Name","Address","Pincode","Latitude", "Longitude"]
    )

In [3]:
def calculate_geodesic_distance(lat1, lon1, lat2, lon2):
    """
    Calculates the high-precision geodesic distance between two points 
    on the WGS84 ellipsoidal surface using pyproj.
    """
    # Load standard WGS84 ellipsoid
    geod = pyproj.Geod(ellps="WGS84")
    
    # pyproj expects coordinates in (Longitude, Latitude) order
    _, _, dist_meters = geod.inv(lon1, lat1, lon2, lat2)
    
    # Convert meters to kilometers
    return dist_meters / 1000.0

In [4]:
def calculate_shortest_road_path(start_coords, end_coords,graph,graph_proj):

    # Extract latitude and longitude from the start and end coordinates
    lat1, lon1 = start_coords[0], start_coords[1]
    lat2, lon2 = end_coords[0], end_coords[1]  

    # Calculate the straight-line (geodesic) distance as a fallback
    geodesic_dis=calculate_geodesic_distance(lat1, lon1, lat2, lon2)

    try:
    # Create Shapely Point objects using (longitude, latitude)
     point1 = Point(lon1, lat1)
     point2 = Point(lon2, lat2)
    
    # Project points into the same coordinate reference system (CRS) as the projected road network graph
     point1_proj, _ = ox.projection.project_geometry(point1, crs=graph.graph['crs'], to_crs=graph_proj.graph['crs'])
     point2_proj, _ = ox.projection.project_geometry(point2, crs=graph.graph['crs'], to_crs=graph_proj.graph['crs'])
    
    # Find the nearest road network nodes to the projected points
     start_node = ox.nearest_nodes(graph_proj, X=point1_proj.x, Y=point1_proj.y) 
     end_node = ox.nearest_nodes(graph_proj, X=point2_proj.x, Y=point2_proj.y)

    # Compute the shortest road path based on edge length
     shortest_route = nx.shortest_path(graph_proj, start_node, end_node, weight="length") 
    
    # Convert the route into a GeoDataFrame and calculate total road distance
     gdf_edges = ox.routing.route_to_gdf(graph_proj, shortest_route)
     actual_road_km = gdf_edges["length"].sum() / 1000 
    
    # Extract route coordinates for map visualization
     route_coordinates = []
     for node in shortest_route:
        node_data = graph.nodes[node]
        route_coordinates.append((node_data['y'], node_data['x']))

    # Return road distance and route coordinates    
     return actual_road_km, route_coordinates
    
    except nx.NetworkXNoPath:
        # Handle cases where no road connection exists between locations
        print("No road path exists between these locations, using geodesic distance.")
        return geodesic_dis,[]
    
    except Exception as e:
        # Handle routing or projection errors and fall back to geodesic distance
        print(f"Road routing unavailable: {e}")
        # Fallback to geodesic distance
        return geodesic_dis,[]


In [5]:
def create_lookup_table(df,graph,graph_proj):
    store_ids = df['Store ID']
    n = len(df)

    matrix = pd.DataFrame(
        np.zeros((n, n)),
        index=store_ids,
        columns=store_ids
    )
    for i in range(n):
        for j in range(i, n):

            if i==j:
                matrix.iloc[i, j]=0
            else:
               start_pt = (df.iloc[i][ 'Latitude'], df.iloc[i]['Longitude'])
               end_pt = (df.iloc[j][ 'Latitude'], df.iloc[j][ 'Longitude'])

               distance,_=calculate_shortest_road_path(start_pt,end_pt,graph,graph_proj)

               matrix.iloc[i, j] = round(distance, 2)
               matrix.iloc[j, i] = round(distance, 2)
    return matrix

In [ ]:
def create_map(hub,x,df,matrix,graph,graph_proj,node_attribute):
    hub_distances = matrix.loc[hub]

    connected = hub_distances[
    (hub_distances <= x) &
    (hub_distances > 0)
     ]
    min_dist = connected.min()
    max_dist = connected.max()
    hub_row = df[df['Store ID']==hub].iloc[0]
    attr_min = df[node_attribute].min()
    attr_max = df[node_attribute].max() 
    tiles ="https://{s}.basemaps.cartocdn.com/rastertiles/voyager_nolabels/{z}/{x}/{y}{r}.png"

    m = folium.Map(
         location=[hub_row['Latitude'],hub_row['Longitude']],
         zoom_start=11,
         tiles=tiles,
         attr='© OpenStreetMap © CARTO'
    )
    value = hub_row[node_attribute]

    if attr_max == attr_min:
          radius = 8
    else:
          normalized = (value - attr_min) / (attr_max - attr_min)
          radius = 5 + normalized * 10
    folium.CircleMarker(
          location=[
          hub_row['Latitude'],
          hub_row['Longitude']
           ],
          radius=radius,
          color="red",
          fill=True,
          fill_color="red",
          fill_opacity=0.7,
          tooltip=(
              f"<b>{hub_row['Store Name']}</b><br>"
              f"{node_attribute}: {value}<br>"
            )
    ).add_to(m)

    folium.Marker(
    location=[hub_row['Latitude'], hub_row['Longitude']],
    icon=folium.DivIcon(
        icon_size=(40, 20),
        icon_anchor=(-10, 5),
        html=f"""
        <div style="
            font-size:10px;
            font-weight:bold;
            color:black;
            background-color:white;
            border:1px solid black;
            border-radius:3px;
            padding:1px 3px;
            white-space: nowrap;
        ">
        {hub_row['Store ID']}
        </div>
        """
    )
).add_to(m)
    for store, distance in connected.items():

      store_row = df[df['Store ID']==store].iloc[0]
      value = store_row[node_attribute]

      if attr_max == attr_min:
          radius = 8
      else:
          normalized = (value - attr_min) / (attr_max - attr_min)
          radius = 5 + normalized * 10

      start_pt = (
        hub_row['Latitude'],
        hub_row['Longitude']
    )

      end_pt = (
        store_row['Latitude'],
        store_row['Longitude']
    )

      _, route_coords = calculate_shortest_road_path(
        start_pt,
        end_pt,
        graph,
        graph_proj
    )
      
      if max_dist == min_dist:
          edge_width = 5
          edge_opacity = 0.8
      else:

          normalized = ( distance - min_dist) / (max_dist - min_dist)
          edge_width =10 - normalized * 6
          edge_opacity = 1 - normalized * 0.7

      if route_coords and len(route_coords) > 0:

        folium.PolyLine(
            locations=route_coords,
            color="blue",
            weight=edge_width,
            opacity=edge_opacity,
            popup=f"Weight: {distance:.2f} km"
        ).add_to(m)

      else:

        folium.PolyLine(
            locations=[start_pt, end_pt],
            color="gray",
            weight=edge_width,
            opacity=edge_opacity,
            dash_array="5,5",
            popup=f"Weight: {distance:.2f} km (geodesic)"
        ).add_to(m)

      folium.CircleMarker(
          location=[
          store_row['Latitude'],
          store_row['Longitude']
           ],
          radius=radius,
          color="green",
          fill=True,
          fill_color="green",
          fill_opacity=0.7,
          tooltip=(
              f"<b>{store_row['Store Name']}</b><br>"
              f"{node_attribute}: {value}<br>"
              f"Distance: {distance:.2f} km"
            )
       ).add_to(m)
      
      folium.Marker(
    location=[store_row['Latitude'], store_row['Longitude']],
    icon=folium.DivIcon(
        icon_size=(40, 20),
        icon_anchor=(-10, 5),
        html=f"""
        <div style="
            font-size:10px;
            font-weight:bold;
            color:black;
            background-color:white;
            border:1px solid black;
            border-radius:3px;
            padding:1px 3px;
            white-space: nowrap;
        ">
        {store_row['Store ID']}
        </div>
        """
    )
).add_to(m)
    file_name = "network_nodeattr_mapping.html"
    m.save(file_name)  

    # Open the generated map in the default web browser             
    webbrowser.open('file://' + os.path.realpath(file_name)) 


In [62]:
if __name__=='__main__':
    data=pd.read_excel('addressList.xlsx')
    location_data=get_coordinates(data)
    location_data['Store ID']=[f'S{i:02d}' for i in range(1,len(location_data)+1)]
    np.random.seed(42)

    location_data["Employees"] = np.random.randint(7, 20, len(location_data))
    location_data["Average Age"] = np.random.randint(21, 45, len(location_data))
    location_data["Phones Sold"] = np.random.randint(80, 401, len(location_data))
    location_data
    print("\n---------------------------FINAL DATAFRAME------------------------------\n")
    display(location_data)

    padding = 0.03                                                 

    # Determine geographic bounding box limits
    north =location_data["Latitude"].max() + padding
    south =location_data["Latitude"].min() - padding
    east =location_data["Longitude"].max() + padding
    west =location_data["Longitude"].min() - padding 
    
    # Download drivable road network within the bounding box
    graph = ox.graph_from_bbox(bbox=(west, south, east, north), network_type="drive")

    # Verify that a valid road network was retrieved
    if len(graph) == 0:
        raise ValueError("No drivable roads found in this specific coordinate region.")
    
    # Project graph to a suitable coordinate reference system for accurate distance calculations
    graph_proj = ox.project_graph(graph)

    # Load or create lookup table
    REBUILD_LOOKUP = False

    if REBUILD_LOOKUP or not os.path.exists("lookup_table.pkl"):
        print("Building Lookup Table.......")
        distance_matrix = create_lookup_table(...)
        distance_matrix.to_pickle("lookup_table.pkl")
    else:
        print("Loading existing Lookup Table.......")
        distance_matrix = pd.read_pickle("lookup_table.pkl")
        
    print("\n-----------------------LOOKUP TABLE---------------------------\n")
    display(distance_matrix)
    hub=str(input("Enter the Store ID of a VI Store: "))
    x=float(input("Enter the threshold distance: "))
    attr = input(
    "Enter node attribute "
    "(Employees / Average Age / Phones Sold): "
)
    print(f"The VI Store ID (Hub) is: {hub}\n")
    print(f"The threshold distance is: {x}\n")
    print(f"The node attribute \n(Employees / Average Age / Phones Sold) is:\n {attr}\n")
    create_map(hub,x,location_data,distance_matrix,graph,graph_proj,attr)


---------------------------FINAL DATAFRAME------------------------------



,Store Name,Address,Pincode,Latitude,Longitude,Store ID,Employees,Average Age,Phones Sold
0,Vi Store Fort,"No 23B, Sir PM Road, Fort",400001,18.941845,72.838958,S01,13,44,100
1,R and R Enterprises (Vi Mini Store),"Shop No 19, Ground Floor, Plot No 66/74, Tabel...",400002,18.946044,72.827683,S02,10,32,246
2,Shravani Enterprises (Vi Mini Store),"Shop No 3, 201, 211 Kesar Building, Princess S...",400002,18.945088,72.826718,S03,19,26,353
3,GP Tel (Vi Mini Store),"Shop No 2, No 156/157, Panju Manzil, IR Road",400003,18.950423,72.837882,S04,17,22,168
4,Angel Telecom (Vi Mini Store),"S 325/A, Plot 323/325, Samual Street, Vadgadi,...",400003,18.951011,72.836021,S05,14,41,395
5,RK Enterprises (Vi Mini Store),"Shop No 2/156/158, Laxmi Niwas, CP Tank Road",400004,18.955389,72.826213,S06,19,21,93
6,Vi Store Parel,"Shop No.6, Kavarana Building, Dr. B A Road, Parel",400012,19.004975,72.839584,S07,11,32,321
7,Vi Store Kurla East,"Shop 02 Usman Gani, Pawar House, Kurla East",400024,19.170749,72.864853,S08,13,42,344
8,Vi Store Santacruz West,"Orchid Pride, G 101, S V Road, Santacruz West",400054,19.076054,72.837972,S09,16,32,132
9,Vi Store Borivali West,"Raichuria House, LT Cross Road, Borivali West",400091,19.231321,72.841312,S10,9,37,171


Loading existing Lookup Table.......

-----------------------LOOKUP TABLE---------------------------



Store ID,S01,S02,S03,S04,S05,S06,S07,S08,S09,S10,...,S13,S14,S15,S16,S17,S18,S19,S20,S21,S22
Store ID,,,,,,,,,,,,,,,,,,,,,
S01,0.00,2.29,2.31,1.72,1.79,2.92,7.55,28.07,16.04,34.59,...,26.20,29.84,21.06,19.79,15.98,2.87,9.66,18.95,8.13,8.64
S02,2.29,0.00,0.79,1.39,1.31,1.67,7.31,27.51,15.48,34.03,...,25.63,29.28,21.08,19.23,16.76,2.40,9.47,19.37,7.57,8.47
S03,2.31,0.79,0.00,2.00,1.92,1.88,7.95,28.15,16.12,34.67,...,26.28,29.92,21.72,19.87,17.40,2.18,10.11,20.01,7.87,9.12
S04,1.72,1.39,2.00,0.00,0.27,1.82,6.76,26.97,14.94,33.49,...,25.10,28.74,20.53,18.69,15.90,3.44,8.92,18.82,7.04,7.92
S05,1.79,1.31,1.92,0.27,0.00,1.58,6.54,26.73,14.70,33.25,...,24.86,28.50,20.30,18.45,15.83,3.32,8.70,18.60,6.80,7.70
S06,2.92,1.67,1.88,1.82,1.58,0.00,6.19,26.39,14.36,32.91,...,24.51,28.16,19.96,18.11,15.64,3.52,8.35,18.25,6.14,7.35
S07,7.55,7.31,7.95,6.76,6.54,6.19,0.00,21.13,9.10,27.65,...,19.26,22.90,14.22,12.85,9.90,9.04,2.61,12.51,3.90,1.61
S08,28.07,27.51,28.15,26.97,26.73,26.39,21.13,0.00,12.06,9.69,...,3.67,4.66,9.95,8.88,18.26,29.46,19.59,14.10,21.93,19.59
S09,16.04,15.48,16.12,14.94,14.70,14.36,9.10,12.06,0.00,18.56,...,10.17,13.82,9.22,3.77,10.12,17.40,7.52,9.63,9.87,7.53


The VI Store ID (Hub) is: S08

The threshold distance is: 8.0

The node attribute 
(Employees / Average Age / Phones Sold) is:
 Employees

